# MATH840 — Lab 7: Seasonal ARIMA

**Week 8 | graded assignment 6 of 7 | 8 points | due 23:59 today**

Your series from Week 4, forecast with **seasonal ARIMA**, $\text{ARIMA}(p,d,q)(P,D,Q)_m$, as the *ARIMA
Models, Part 2* lecture defines it. **Specify the model by hand first** — the differences from the tests,
$p, q$ from the first lags and $P, Q$ from the seasonal lags of the ACF and PACF — and only then compare
with `AutoARIMA`. ARIMA only: no ETS, no regression, no averaging.

**Accuracy is marked again.** Your forecast is scored on the hidden horizon against your own benchmark:
beat it — 3 points, within 10% — 2, worse — 1 with a written explanation.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | EDA and visualisation | 1 |
| 2 | Forecast accuracy against your benchmark (scored from your file) | 3 |
| 3 | Justification block | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab08_forecast.csv`, `SURNAME_lab08.pdf` and `SURNAME_lab08.ipynb` to the Week 8
activity on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked.

In [ ]:
!pip install -q statsforecast==2.1.1   # the version the pool was screened with

import hashlib
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import ARIMA, AutoARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier produces the same series as in Weeks 4, 6 and 7. If the code printed below is not the
one from your earlier submissions, stop and check `MY_ID`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)
dates = s["ds"]

print(f"series {CODE}: {len(s)} observations, {dates.min().date()} to {dates.max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")

### Helpers

Given, and the same as in the practice session and Lab 6. The benchmark is computed by the scorer's rule.
`fit_arima` builds every model in this lab: a hand specification `(p, d, q)` with seasonal `(P, D, Q)`,
or `AutoARIMA` when `order=None`. With `log=True` it fits on $\log y$ and returns forecasts and intervals
on the original scale.

In [ ]:
def benchmark_forecasts(history: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    # The four simple methods from Week 1.
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


train, valid = y[:-H], y[-H:]
train_dates, valid_dates = dates.iloc[:-H], dates.iloc[-H:]

simple = benchmark_forecasts(train, H, M)
simple_scores = pd.DataFrame({name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                              for name, fc in simple.items()}).T
BENCHMARK = simple_scores["rmse"].idxmin()
print(simple_scores.round(3).to_string())
print(f"\nyour benchmark: {BENCHMARK} (validation MASE {simple_scores.loc[BENCHMARK, 'mase']:.3f})")

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())

In [ ]:
def difference_plan(series: np.ndarray, m: int, log: bool = False):
    # Step 1: D from seasonal strength, then d from KPSS on what is left. Returns D, d and the differenced series.
    x = np.log(series) if log else np.asarray(series, float)
    D = nsdiffs(x, period=m)
    z = x[m:] - x[:-m] if D else x.copy()
    d = ndiffs(z)
    z = np.diff(z, n=d) if d else z
    print(f"{'log, ' if log else ''}D = {D}, d = {d}: {len(x)} observations -> {len(z)} after differencing")
    return D, d, z


def acf_pacf(z: np.ndarray, lags: int, title: str = ""):
    # Step 2: read p, q at lags 1, 2, 3 and P, Q at lags m, 2m, 3m (shaded).
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    plot_acf(z, lags=lags, zero=False, ax=axes[0], auto_ylims=True, title=f"{title} ACF")
    plot_pacf(z, lags=lags, zero=False, ax=axes[1], auto_ylims=True, title=f"{title} PACF")
    for ax in axes:
        for k in range(M, lags + 1, M):
            ax.axvline(k, color="#e64173", alpha=0.25, linewidth=6)
    plt.show()


class Fitted:
    # A fitted ARIMA whose forecasts are on the original scale, whatever it was fitted on.
    def __init__(self, model, log):
        self.model, self.log = model, log
        self.model_ = model.model_

    def predict(self, h: int, level=None):
        out = self.model.predict(h=h, level=level)
        return {k: np.exp(v) for k, v in out.items()} if self.log else out


def fit_arima(history: np.ndarray, order=None, seasonal=(0, 0, 0), log: bool = False,
              drift: bool = False) -> Fitted:
    # order=(p, d, q) and seasonal=(P, D, Q) for a hand specification; order=None lets AutoARIMA search.
    x = np.log(history) if log else np.asarray(history, float)
    if order is None:
        model = AutoARIMA(season_length=M)
    else:
        model = ARIMA(order=tuple(order), seasonal_order=tuple(seasonal), season_length=M,
                      include_drift=drift and (order[1] + seasonal[1]) == 1)
    model.fit(y=x)
    return Fitted(model, log)


def compare(candidates: dict, history: np.ndarray, lb_lags: int | None = None) -> pd.DataFrame:
    # Step 3: every candidate fitted on `history`; AICc and Ljung-Box with model_df = p + q + P + Q.
    # AICc is comparable only between candidates with the same d, D and log choice.
    lb_lags = lb_lags or 2 * M
    rows = []
    for name, spec in candidates.items():
        fit = fit_arima(history, **spec)
        p, q, P, Q = fit.model_["arma"][:4]
        resid = pd.Series(fit.model_["residuals"])
        rows.append({"candidate": name,
                     "model": ("log " if fit.log else "") + str(ARIMASummary(fit.model_)).strip(),
                     "AICc": fit.model_["aicc"],
                     f"Ljung-Box p ({lb_lags})": acorr_ljungbox(resid, lags=[lb_lags], model_df=p + q + P + Q)["lb_pvalue"].iloc[0],
                     f"resid acf({M})": resid.autocorr(M),
                     "valid MASE": mase(valid, fit_arima(train, **spec).predict(h=H)["mean"])})
    return pd.DataFrame(rows)


def rolling_mase(spec: dict, history: np.ndarray, h: int, windows: int = 3) -> float:
    # Step 4: mean MASE over `windows` origins on the original scale; the last window is the validation window.
    errors = []
    for k in range(windows):
        end = len(history) - k * h
        fit = fit_arima(history[:end - h], **spec)
        errors.append(mase(history[end - h:end], fit.predict(h=h)["mean"]))
    return float(np.mean(errors))

## 1. EDA and visualisation

*1 point.* The plots that justify the transformation and the differencing:

- a time plot, read for trend, season, and whether the seasonal swing grows with the level (`log=True`);
- the validation split on a plot, with your chosen model's forecast and 80% interval over it;
- one sentence per plot. A plot with no reading attached earns nothing.

In [ ]:
# TODO: time plot (and anything else your argument uses), with readings

**What the plots show.**

→

## 2. The model

*3 points, scored from your submission file* on the hidden horizon against your benchmark:
skill = your MASE / your benchmark's MASE; below 1 — 3 points, up to 1.1 — 2, above — 1.

### 2.1 Differencing

In [ ]:
LOG = False                         # <-- True if the seasonal swing grows with the level
D, d, z = difference_plan(y, M, log=LOG)

# TODO: plot z, the differenced series

### 2.2 Specification by hand, then `AutoARIMA`

Read the ACF and PACF of `z` **before** running anything automatic: $p, q$ at the first lags, $P, Q$ at
the shaded seasonal lags. Write down at least two candidates, add `AutoARIMA`, compare with `compare`, and
decide with `rolling_mase` — with your benchmark in the same table.

A candidate is a dict of `fit_arima`'s arguments, e.g.
`{"order": (0, d, 1), "seasonal": (0, D, 1), "log": LOG}` — the airline model.

In [ ]:
acf_pacf(z, lags=3 * M, title="differenced series:")

**Your reading: which lags at the start, which at the seasonal lags, which candidates.**

→

In [ ]:
candidates = {
    # TODO: your hand-specified candidates, from the reading above
    # "by hand: airline": {"order": (0, d, 1), "seasonal": (0, D, 1), "log": LOG},
    "AutoARIMA": {"order": None, "log": LOG},
}
table = compare(candidates, y)
table["rolling MASE (3 windows)"] = [rolling_mase(spec, y, H) for spec in candidates.values()]
table

### 2.3 Residuals and interval on the validation window

The residuals of your chosen model (ACF to lag $3m$, Ljung-Box with $p + q + P + Q$) and its 80% interval
over the validation window: how many of the `H` points does it cover?

In [ ]:
# TODO: residual diagnostics and interval coverage of your chosen model

### 2.4 The submission

Refit your chosen candidate on the **whole** history `y` and forecast `H` steps with the model's 80%
interval. If you chose `AutoARIMA`, it searches again on all of `y` — print what it chose.

In [ ]:
# TODO: your final forecast, fitted on all of y.
#
#   final = fit_arima(y, **candidates["..."])
#   fc = final.predict(h=H, level=[80])
YHAT = None          # np.array of length H  (fc["mean"])
YHAT_LO_80 = None    # np.array of length H  (fc["lo-80"])
YHAT_HI_80 = None    # np.array of length H  (fc["hi-80"])

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H:
        raise ValueError(f"{name} must be an array of length H = {H}")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own cells,
not adjectives.

**3.1 The specification, before the machine** *(1)* — $D$, $d$ and the log decision from the tests and
plots; the lags at the start of the ACF/PACF that named $p, q$, and the seasonal lags that named $P, Q$.

→

**3.2 What `AutoARIMA` chose, and what you submitted** *(1)* — where it agrees or disagrees with you and
why; your candidates with AICc and rolling MASE; the model you submitted and the evidence for it. If it is
not `AutoARIMA`'s choice, say what made you leave it.

→

**3.3 What is left, and what you expect** *(1)* — Ljung-Box with $p + q + P + Q$ degrees of freedom, read;
the residual autocorrelation at lag $m$ compared with your Lab 6 model; the interval's coverage on the
validation window; your expected skill against the benchmark. If you expect to lose, say why.

→

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: candidates are compared on `train`/`valid` and `rolling_mase`; only the final model sees all
  of `y`.
- Every number in your text comes from a cell.

## 5. Submit

Run both cells, then upload **three files** to the Week 8 activity on Moodle before 23:59:

- `SURNAME_lab08_forecast.csv` — written below;
- `SURNAME_lab08.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab08.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab08_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")